In [1]:
%%prun -s cumulative -q -l 15 -T profile_results.txt 

import os
import numpy as np
import matplotlib.pyplot as plt
import sys
import logging

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('')
# Assuming the notebook is in a 'notebooks' directory at the same level as 'src', 'config', etc.
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) 
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# Make sure to import the HDF5 versions of the functions
from src.utils.validation_utils import (
    calculate_metrics_for_experiment_hdf5,    
    generate_roc_pr_data_gt_velocity_variant_hdf5, 
    load_config_from_hdf5 
)
from src.visualization.metrics_plots import plot_roc_curve, plot_precision_recall_curve
from src.core.constants import OcclusionResult

# --- Configuration ---
GT_LABELS_BASE_DIR_HDF5 = "/home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated"
MDET_RESULTS_ROOT_DIR = "/home/drugge/staff-umbrella/TeamHolgerResearch/drugge/m_detector_output"
mdet_experiment_dir_path_hdf5 = MDET_RESULTS_ROOT_DIR

# Base eval params
base_eval_params = {
    "mdet_label_field_name": "mdet_label",
    "mdet_dynamic_label_value": OcclusionResult.OCCLUDING_IMAGE.value,
    "coordinate_tolerance_for_verification": 1e-3,
    # Default range parameters (can be overridden by scene's HDF5 config)
    "mdet_min_point_range_meters": 1.0,
    "mdet_max_point_range_meters": 80.0,
}

print(f"  MDet HDF5 Path: {mdet_experiment_dir_path_hdf5}")
print(f"  GT HDF5 Path:   {GT_LABELS_BASE_DIR_HDF5}")


# --- 1. Calculate Overall Metrics (using HDF5 files) ---
eval_params_for_summary = base_eval_params.copy()
eval_params_for_summary["gt_velocity_threshold"] = 1.0 

experiment_summary_hdf5 = calculate_metrics_for_experiment_hdf5( 
    mdet_experiment_dir=mdet_experiment_dir_path_hdf5,
    gt_labels_base_dir=GT_LABELS_BASE_DIR_HDF5,
    eval_params=eval_params_for_summary
)

if experiment_summary_hdf5:
    print(f"\n--- Experiment Summary Metrics (HDF5, GT Vel Thresh = {eval_params_for_summary['gt_velocity_threshold']} m/s) ---")
    print(f"  File Format Used: {experiment_summary_hdf5.get('file_format_used', 'N/A')}")
    print(f"  Precision: {experiment_summary_hdf5['Precision']:.4f}")
    print(f"  Recall:    {experiment_summary_hdf5['Recall']:.4f}")
    print(f"  F1 Score:  {experiment_summary_hdf5['F1']:.4f}")
    print(f"  IoU Dynamic: {experiment_summary_hdf5['overall_iou_dynamic']:.4f}")
    print(f"  Scenes Evaluated: {experiment_summary_hdf5['num_scenes_successfully_evaluated']}/{experiment_summary_hdf5['num_scenes_total_in_dir']}")
else:
    print("Failed to get experiment summary using HDF5 files.")


Added project root to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python
  MDet HDF5 Path: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/m_detector_output
  GT HDF5 Path:   /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated


  Scenes (HDF5): 100%|██████████| 1/1 [00:36<00:00, 36.23s/it]



--- Experiment Summary Metrics (HDF5, GT Vel Thresh = 1.0 m/s) ---
  File Format Used: hdf5
  Precision: 0.2587
  Recall:    0.0862
  F1 Score:  0.1294
  IoU Dynamic: 0.0692
  Scenes Evaluated: 1/1
 
*** Profile printout saved to text file 'profile_results.txt'. 


In [ ]:
import os
print(os.getcwd())

In [ ]:

# --- 2. Generate Data for ROC-like and PR-like Curves by Varying GT Velocity Threshold ---
# Define a range of GT velocity thresholds to test.
# Example: 0.05 m/s (very sensitive to any motion) to 2.0 m/s (only very fast objects are "dynamic")
gt_velocity_thresholds_for_curve = [0.05] #

print("\nGenerating ROC-like and PR-like data by varying GT velocity threshold...")
roc_like_data = generate_roc_pr_data_gt_velocity_variant(
    mdet_experiment_dir=mdet_experiment_dir_path,
    gt_labels_base_dir=GT_LABELS_BASE_DIR,
    eval_params_base=base_eval_params, # Pass the base_eval_params
    gt_velocity_thresholds_for_roc=gt_velocity_thresholds_for_curve,
    experiment_summary=experiment_summary
)

if roc_like_data:
    # Sort by FPR for a standard ROC plot appearance if needed, though varying GT vel doesn't guarantee FPR monotonicity
    # For now, we'll plot as is, or you can sort if the FPR values are not monotonic.
    # If sorting is desired:
    # sorted_indices = np.argsort(roc_like_data['fpr'])
    # fpr_sorted = np.array(roc_like_data['fpr'])[sorted_indices]
    # tpr_sorted = np.array(roc_like_data['tpr'])[sorted_indices]
    # precision_sorted = np.array(roc_like_data['precision'])[sorted_indices] # Sort precision by same indices if plotting vs FPR
    # recall_sorted = np.array(roc_like_data['recall'])[sorted_indices] # Recall is TPR

    # Plot "ROC-like" Curve
    plt.figure(figsize=(7,6))
    plot_roc_curve( 
        fpr_list=roc_like_data['fpr'], # or fpr_sorted
        tpr_list=roc_like_data['tpr'], # or tpr_sorted
        experiment_label=f"{experiment_id_to_evaluate} (varying GT vel_thresh)",
        ax=plt.gca()
    )
    plt.title(f"ROC-like Curve (Varying GT Velocity Threshold) - {experiment_id_to_evaluate}")
    plt.show()

    # Plot "Precision-Recall-like" Curve
    # For PR curves, typically recall is on X-axis, precision on Y.
    # We can sort by recall (which is TPR) for standard PR curve plotting.
    # sorted_pr_indices = np.argsort(roc_like_data['recall'])
    # recall_pr_sorted = np.array(roc_like_data['recall'])[sorted_pr_indices]
    # precision_pr_sorted = np.array(roc_like_data['precision'])[sorted_pr_indices]
    
    plt.figure(figsize=(7,6))
    plot_precision_recall_curve(
        recall_list=roc_like_data['recall'], # or recall_pr_sorted
        precision_list=roc_like_data['precision'], # or precision_pr_sorted
        experiment_label=f"{experiment_id_to_evaluate} (varying GT vel_thresh)",
        ax=plt.gca()
    )
    plt.title(f"Precision-Recall-like Curve (Varying GT Velocity Threshold) - {experiment_id_to_evaluate}")
    plt.show()

    # Optional: Plot TPR/FPR/Precision directly against the GT Velocity Threshold
    plt.figure(figsize=(10, 7))
    plt.plot(roc_like_data['gt_velocity_thresholds'], roc_like_data['tpr'], label='TPR (Recall)', marker='o', linestyle='-')
    plt.plot(roc_like_data['gt_velocity_thresholds'], roc_like_data['fpr'], label='FPR', marker='x', linestyle='-')
    plt.plot(roc_like_data['gt_velocity_thresholds'], roc_like_data['precision'], label='Precision', marker='s', linestyle='-')
    plt.xlabel("Ground Truth Velocity Threshold (m/s)")
    plt.ylabel("Metric Value")
    plt.title(f"Metrics vs. GT Velocity Threshold - {experiment_id_to_evaluate}")
    plt.legend()
    plt.grid(True)
    plt.show()

else:
    print("Failed to generate ROC-like/PR-like data by varying GT velocity threshold.")

print("\nMetrics calculation notebook finished.")